# EPOCH 5th Datathon — 개선 노트북 06 (YouTube Rising Channel)

05번 노트북(리더보드 **0.4554**)을 기반으로 다음을 고쳤다.

| # | 개선 | 이유 |
|---|---|---|
| 1 | **view_5d 측정 시점 마스킹** | 09-01에는 기준일 5.5일(132h) 전 이후 영상의 view_5d가 없다(코드북 §2.4). 05번은 학습 기준일에선 그 영상들을 썼기 때문에 `med_7`·`mom_7` 등이 학습/예측에서 다른 값이었다 |
| 2 | **라벨 재현 검증** | 자체 26일 라벨이 공식 규칙과 같은지 08-11·08-25 공식 라벨과 대조 |
| 3 | **검증 라벨 창 통일** | 검증은 18일 창 자체 라벨(07-26~08-07)로 학습 → 18일 창 공식 08-25로 평가 |
| 4 | **신규 피처** | 업로드율·미래 무업로드 확률·상방 여지·출시 예정 게임 |
| 5 | **growth 회귀(Ridge)** | 0/1로 자르며 버리던 정보를 활용 |
| 6 | **라벨 생성 시뮬레이션 점수** | 업로드 수 × 조회수 분산 × 낮은 기준선을 직접 모사 (학습 불필요) |
| 7 | **블렌드 후보 비교 + 부트스트랩** | 694행 잡음(±0.03)을 감안해 차이의 신뢰도 확인 |
| 8 | **동점 깨기** | 순위 평균으로 생기던 동점 제거 |

> 실행 환경: Python 3, duckdb, pandas, numpy, scipy, scikit-learn, matplotlib.  
> 데이터 경로 `../data/raw/epoch_data`, 결과 저장 `../submissions` (05번과 동일).

In [ ]:
import os, glob, platform, warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')
import logging; logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
plt.rcParams['font.family'] = {'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW     = os.environ.get('EPOCH_RAW', '../data/raw/epoch_data')
SUB_DIR = os.environ.get('EPOCH_SUB', '../submissions')
D       = '2026-09-01'   # 본선 기준일
FUT     = 26             # 본선 정답 구간 (일)
FUT_OFF = 18             # 공식 train_labels 정답 구간 (일)
LAG_H   = 132            # view_5d가 확정되기까지 걸리는 시간 (업로드 후 132h)
SEED    = 42
os.makedirs(SUB_DIR, exist_ok=True)

con = duckdb.connect()
TABLES = []
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")
    TABLES.append(name)
print(TABLES)
con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df()

## 1. 라벨 — 공식 규칙 재현 + 검증

규칙: 롱폼만, 과거 view_5d 3편 이상, 과거 중앙값 ≥ 100, `growth = log1p(future_med) − log1p(past_med)`, 기준일 내 상위 20%.  
`past_days`로 과거 구간을 전체 이력(None) 또는 최근 N일로 바꿔 볼 수 있다.

**재현 검증 아이디어**: 대상 채널 집합(3편 이상·100 이상)은 *과거 데이터만으로* 결정되므로, 우리가 가진 데이터로 공식 08-11·08-25 행 집합을 **정확히** 재현할 수 있어야 한다.  
08-11은 정답 구간(08-11~08-29)도 거의 다 있으므로 target 일치율까지 본다(08-26 이후 영상은 view_5d가 없어 약간 어긋날 수 있음).

In [ ]:
def make_labels(Dref, fut=FUT, past_days=None):
    past_lo = f"AND published_at >= TIMESTAMP '{Dref}' - INTERVAL {past_days} DAY" if past_days else ""
    g = con.sql(f'''
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}' {past_lo}) AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}' {past_lo}) AS n_past,
                 COUNT(view_5d)  FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                                           AND published_at <  TIMESTAMP '{Dref}' + INTERVAL {fut} DAY) AS n_future,
                 COALESCE(MEDIAN(view_5d) FILTER (
                     WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at <  TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) AS future_score
          FROM video_5d_views
          WHERE is_shorts = 0 AND view_5d IS NOT NULL
          GROUP BY 1
        ),
        scored AS (
          SELECT channel_id, n_future, LN(1 + future_score) - LN(1 + past_score) AS growth
          FROM agg WHERE n_past >= 3 AND past_score >= 100
        )
        SELECT channel_id, n_future, growth,
               CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored
    ''').df()
    g['ref_date'] = Dref
    return g

tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
tl['ref_date']   = tl.row_id.str.rsplit('_', n=1).str[1]

rows = []
for pdays in [None, 28, 14]:
    for d in ['2026-08-11', '2026-08-25']:
        own = make_labels(d, FUT_OFF, pdays)
        off = tl[tl.ref_date == d]
        a, b = set(own.channel_id), set(off.channel_id)
        m = off.merge(own, on='channel_id', suffixes=('_off', '_own'))
        rows.append({'past_days': str(pdays), 'ref_date': d, '공식 행': len(b), '재현 행': len(a),
                     '행 집합 Jaccard': round(len(a & b) / len(a | b), 4),
                     'target 일치율': round((m.target_off == m.target_own).mean(), 4) if d == '2026-08-11' else np.nan})
repro = pd.DataFrame(rows)
display(repro)

score = repro.groupby('past_days').agg(j=('행 집합 Jaccard', 'mean'), t=('target 일치율', 'mean'))
best = score.sort_values(['j', 't'], ascending=False).index[0]
PAST_DAYS = None if best == 'None' else int(best)
print('채택 past_days =', PAST_DAYS, '| 평균 Jaccard =', round(score.loc[best, 'j'], 4))
if score.loc[best, 'j'] < 0.99:
    print('⚠ 행 집합이 공식과 완전히 일치하지 않음 — 공식 규칙과 다른 부분이 있을 수 있으니 확인 필요')

## 2. 피처 — `make_features(Dref, lag_h)`

**핵심 수정 (마스킹)**: view_5d를 쓰는 모든 피처는 `published_at < Dref − 132h`인 영상만 쓴다.  
09-01 배포 데이터에는 그 이후 영상의 view_5d가 원래 없으므로, 학습 기준일에서도 똑같이 가려야 학습·예측 피처가 같은 의미가 된다.
최근 창(7·14일)은 "측정 가능한 마지막 시점" 기준으로 다시 정의했다. 업로드 수 피처(`videos`)는 업로드 즉시 알 수 있으므로 가리지 않는다.

**신규 피처**

| 피처 | 의미 |
|---|---|
| `rate_28` | 최근 28일(이력이 짧으면 이력 길이) 롱폼 업로드율 /일 |
| `exp_future_n` | 정답 구간 동안 예상 롱폼 수 = rate × 구간 길이 |
| `p_no_post` | 정답 구간에 롱폼이 하나도 없을 확률 (푸아송) → future=0 → 무조건 하위 |
| `upside` | `log_q90 − log_past_med` : 가끔 터지는 폭 |
| `n_meas_14` | 측정 가능한 최근 14일 롱폼 수 |
| `n_upcoming_games` | 다뤄 온 게임 중 정답 구간에 출시 예정인 게임 수 (Steam·IGDB) |
| `n_recent_release` | 다뤄 온 게임 중 기준일 직전 14일에 출시된 게임 수 |

In [ ]:
def _parse_dates(s):
    try:
        return pd.to_datetime(s, errors='coerce', format='mixed')
    except (TypeError, ValueError):
        return pd.to_datetime(s, errors='coerce')

# 게임 출시일 (Steam + IGDB) → 영상-게임-출시일 테이블
_rel = []
if {'apps', 'video_games'} <= set(TABLES):
    a = con.sql("SELECT app_id, CAST(release_date AS VARCHAR) AS rd FROM apps").df()
    a['release'] = _parse_dates(a.rd)
    m = con.sql("SELECT CAST(video_id AS VARCHAR) AS video_id, app_id FROM video_games").df().merge(a, on='app_id')
    _rel.append(pd.DataFrame({'video_id': m.video_id, 'game': 's' + m.app_id.astype(str), 'release': m.release}))
if {'igdb_games', 'video_igdb_games'} <= set(TABLES):
    g = con.sql("SELECT igdb_id, CAST(first_release_date AS VARCHAR) AS rd FROM igdb_games").df()
    g['release'] = _parse_dates(g.rd)
    m = con.sql("SELECT CAST(video_id AS VARCHAR) AS video_id, igdb_id FROM video_igdb_games").df().merge(g, on='igdb_id')
    _rel.append(pd.DataFrame({'video_id': m.video_id, 'game': 'i' + m.igdb_id.astype(str), 'release': m.release}))
GAME_REL = pd.concat(_rel, ignore_index=True).dropna(subset=['release']) if _rel else pd.DataFrame(columns=['video_id', 'game', 'release'])
con.register('game_rel', GAME_REL)
print('출시일이 있는 영상-게임 매핑:', len(GAME_REL))


def game_feats(Dref, fut):
    if len(GAME_REL) == 0:
        return pd.DataFrame(columns=['channel_id', 'n_upcoming_games', 'n_recent_release'])
    return con.sql(f'''
        WITH cg AS (
            SELECT DISTINCT v.channel_id, r.game, r.release
            FROM videos v JOIN game_rel r ON CAST(v.video_id AS VARCHAR) = r.video_id
            WHERE v.published_at <  TIMESTAMP '{Dref}'
              AND v.published_at >= TIMESTAMP '{Dref}' - INTERVAL 60 DAY
        )
        SELECT channel_id,
               COUNT(DISTINCT game) FILTER (WHERE release >= TIMESTAMP '{Dref}'
                                              AND release <  TIMESTAMP '{Dref}' + INTERVAL {fut} DAY)  AS n_upcoming_games,
               COUNT(DISTINCT game) FILTER (WHERE release >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY
                                              AND release <  TIMESTAMP '{Dref}')                        AS n_recent_release
        FROM cg GROUP BY 1
    ''').df()


def make_features(Dref, fut=FUT, lag_h=LAG_H, past_days=None):
    meas_cut = f"TIMESTAMP '{Dref}' - INTERVAL {lag_h} HOUR"     # view_5d를 알 수 있는 마지막 업로드 시각
    past_lo  = f"AND published_at >= TIMESTAMP '{Dref}' - INTERVAL {past_days} DAY" if past_days else ""
    f = con.sql(f'''
    WITH lv AS (
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago,
               DATE_DIFF('hour', published_at, {meas_cut}) / 24.0      AS age        -- 측정 가능 시점 기준 경과일
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < {meas_cut} {past_lo}
    ),
    v5 AS (
        SELECT channel_id,
               COUNT(*)                                     AS n_long_5d,
               COUNT(*) FILTER (WHERE age < 14)             AS n_meas_14,
               MEDIAN(view_5d)                              AS past_med,
               MEDIAN(view_5d) FILTER (WHERE age < 7)       AS med_7,
               MEDIAN(view_5d) FILTER (WHERE age < 14)      AS med_14,
               MEDIAN(view_5d) FILTER (WHERE age >= 14)     AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))         AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))      AS comment_rate,
               STDDEV(LN(1 + view_5d))                      AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)          AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)          AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)       AS trend_slope,
               MIN(days_ago)                                AS days_since_meas
        FROM lv GROUP BY 1
    ),
    sv AS (
        SELECT channel_id, MEDIAN(view_5d) AS short_med
        FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < {meas_cut}
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0)                                                  AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0  AS days_since_long,
               DATE_DIFF('hour', MIN(published_at), TIMESTAMP '{Dref}') / 24.0                               AS days_hist
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1
    )
    SELECT * FROM up LEFT JOIN v5 USING (channel_id) LEFT JOIN sv USING (channel_id)
                     LEFT JOIN games USING (channel_id)
    ''').df()

    f = f.merge(game_feats(Dref, fut), on='channel_id', how='left')
    for c in ['n_upcoming_games', 'n_recent_release', 'n_games', 'n_long_5d', 'n_meas_14']:
        f[c] = f[c].fillna(0)

    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    f['rate_28']           = f.n_long_28 / f.days_hist.clip(1, 28)
    f['exp_future_n']      = f.rate_28 * fut
    f['p_no_post']         = np.exp(-f.exp_future_n)
    f['upside']            = f.log_q90 - lp
    return f

BASE = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
        'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
        'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
        'log_short_med_rel', 'log_n_long_5d']
NEW  = ['rate_28', 'exp_future_n', 'p_no_post', 'upside', 'n_meas_14', 'n_upcoming_games', 'n_recent_release']
PLUS = BASE + NEW
print(len(BASE), '→', len(PLUS), 'features')

### 2.1. 마스킹이 왜 필요한가 — 학습/예측 피처 비교

05번 방식(`lag_h=0`)과 마스킹(`lag_h=132`)에서, 학습 기준일과 09-01의 피처 분포를 비교한다.  
05번 방식에서 학습 기준일과 09-01의 `med_7` 결측률·`days_since_meas`가 크게 다르면, 그만큼 학습한 관계가 예측 때 깨졌다는 뜻이다.

In [ ]:
chk = []
for lag in [0, LAG_H]:
    for d in ['2026-07-30', '2026-08-25', D]:
        f = make_features(d, lag_h=lag, past_days=PAST_DAYS)
        chk.append({'방식': '05번 (마스킹 없음)' if lag == 0 else '06번 (132h 마스킹)', '기준일': d,
                    'med_7 결측률': round(f.med_7.isna().mean(), 3),
                    'days_since_meas 중앙값': round(f.days_since_meas.median(), 2),
                    'mom_7 표준편차': round(f.mom_7.std(), 3)})
display(pd.DataFrame(chk))

## 3. 학습·검증 데이터

| 용도 | 라벨 | 기준일 | 비고 |
|---|---|---|---|
| **검증 학습** | 자체 **18일** 라벨 | 07-26 ~ 08-07 | 정답 구간이 08-25에 끝남 → 08-25 평가와 겹치지 않음 |
| **검증 평가** | 공식 08-25 (18일) | 08-25 | 학습과 같은 창 길이 |
| **최종 학습** | 자체 **26일** 라벨 + 공식 08-11·08-25 | 07-26 ~ 08-05 | 본선과 같은 창 길이 |

05번은 검증 학습에 26일 라벨(07-26~07-30)을 쓰고 18일 라벨로 평가했다. 이번에는 창 길이를 맞추고 학습 기준일도 5개 → 13개로 늘렸다.  
참고: 26일 자체 라벨 중 07-31 이후 기준일은 정답 구간 끝 며칠의 view_5d가 아직 없어 실제 창이 조금 짧다.

In [ ]:
def build(labels, fut):
    return pd.concat([labels[labels.ref_date == d].merge(make_features(d, fut=fut, past_days=PAST_DAYS), on='channel_id', how='left')
                      for d in sorted(labels.ref_date.unique())], ignore_index=True)

dates = lambda a, b: pd.date_range(a, b, freq='D').strftime('%Y-%m-%d').tolist()

val_tr = build(pd.concat([make_labels(d, FUT_OFF, PAST_DAYS) for d in dates('2026-07-26', '2026-08-07')]), FUT_OFF)
val_va = build(tl[tl.ref_date == '2026-08-25'][['channel_id', 'ref_date', 'target']], FUT_OFF)

own26  = build(pd.concat([make_labels(d, FUT, PAST_DAYS) for d in dates('2026-07-26', '2026-08-05')]), FUT)
off    = build(tl[['channel_id', 'ref_date', 'target']], FUT)     # 최종 학습용: 예측 구간(26일) 기준 피처
final_tr = pd.concat([own26, off], ignore_index=True)

print(f'검증 학습 {len(val_tr)}행 | 검증 평가 {len(val_va)}행 | 최종 학습 {len(final_tr)}행 (자체26 {len(own26)} + 공식 {len(off)})')

## 4. 모델 구성 요소

- **lr / lr_base** — 로지스틱 회귀 (PLUS / BASE 피처, C=0.1, class_weight=balanced)
- **rule** — `−log_past_med` (평균 회귀)
- **ridge** — growth 자체를 회귀. 미래에 영상이 없어 growth가 크게 음수가 되는 행은 −2.5에서 자른다
- **sim** — 라벨 생성 시뮬레이션 (아래)

### 4.1. 시뮬레이션 점수

채널마다 측정 가능한 최근 롱폼 log 조회수 분포와 업로드율 λ를 구한 뒤 N회 반복한다.
1. 정답 구간 롱폼 수 `k ~ Poisson(λ × 구간 길이)` — k=0이면 future=0
2. k>0이면 과거 분포에서 k개를 뽑아(약간의 잡음 추가) 중앙값 → growth
3. 매 반복마다 전체 채널의 growth 상위 20% 임계값을 넘으면 Rising

`score = Rising으로 뽑힌 비율`. 학습이 없으니 과적합이 없고, 구간 길이(18/26일)를 그대로 반영한다.

In [ ]:
def logreg(C=0.1):
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=3000, C=C, class_weight='balanced'))])

def ridge(alpha=10.0):
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('m', Ridge(alpha=alpha))])

GROWTH_CLIP = -2.5

def rank01(x):
    x = pd.Series(np.asarray(x, dtype=float))
    return rankdata(x.fillna(x.median())) / len(x)


def sim_score(Dref, feats, horizon, n_sim=500, jitter=0.3, recent_days=28, seed=SEED):
    rng = np.random.default_rng(seed)
    lv = con.sql(f'''
        SELECT channel_id, LN(1 + view_5d) AS lv,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL
          AND published_at < TIMESTAMP '{Dref}' - INTERVAL {LAG_H} HOUR
    ''').df()
    groups = {k: g for k, g in lv.groupby('channel_id')}
    G = np.full((n_sim, len(feats)), np.nan)
    for j, r in enumerate(feats[['channel_id', 'log_past_med', 'rate_28']].itertuples(index=False)):
        g = groups.get(r.channel_id)
        if g is None or pd.isna(r.log_past_med):
            continue
        L = g.lv[g.days_ago < recent_days + LAG_H / 24].values
        if len(L) < 3:
            L = g.lv.values
        lam = 0.0 if pd.isna(r.rate_28) else r.rate_28 * horizon
        ks = rng.poisson(lam, n_sim)
        growth = np.full(n_sim, -r.log_past_med)                 # 영상 없음 → future = 0
        for k in np.unique(ks[ks > 0]):
            idx = np.where(ks == k)[0]
            s = rng.choice(L, size=(len(idx), k)) + rng.normal(0, jitter, (len(idx), k))
            growth[idx] = np.median(s, axis=1) - r.log_past_med
        G[:, j] = growth
    thr = np.nanquantile(G, 0.8, axis=1, keepdims=True)
    return np.nanmean(G >= thr, axis=0) if np.isfinite(G).any() else np.zeros(len(feats))


def components(train, test, Dref, horizon, ridge_train=None):
    out = {}
    out['lr_base'] = logreg().fit(train[BASE], train.target).predict_proba(test[BASE])[:, 1]
    out['lr']      = logreg().fit(train[PLUS], train.target).predict_proba(test[PLUS])[:, 1]
    rt = train if ridge_train is None else ridge_train
    rt = rt[rt.growth.notna()]
    out['ridge']   = ridge().fit(rt[PLUS], rt.growth.clip(lower=GROWTH_CLIP)).predict(test[PLUS])
    out['rule']    = -test.log_past_med.values
    out['sim']     = sim_score(Dref, test, horizon)
    return out


BLENDS = {
    'B0_05방식(lr_base+rule)':  {'lr_base': 1, 'rule': 1},
    'B1_lr+rule':               {'lr': 1, 'rule': 1},
    'B2_lr+rule+ridge':         {'lr': 1, 'rule': 1, 'ridge': 1},
    'B3_lr+rule+sim':           {'lr': 1, 'rule': 1, 'sim': 1},
    'B4_lr+rule+ridge+sim':     {'lr': 1, 'rule': 1, 'ridge': 1, 'sim': 1},
    'B5_ridge+sim':             {'ridge': 1, 'sim': 1},
}

def blend(comp, w):
    return sum(v * rank01(comp[k]) for k, v in w.items()) / sum(w.values())

## 5. 검증 — 공식 08-25 홀드아웃

블렌드 후보마다 PR-AUC와, B0(05번 방식) 대비 차이를 **짝지은 부트스트랩**(1,000회)으로 본다.
`P(개선)`이 0.5 근처면 차이는 잡음일 가능성이 크다.

In [ ]:
vc = components(val_tr, val_va, '2026-08-25', FUT_OFF)
y = val_va.target.values
rng = np.random.default_rng(SEED)
boot = [rng.integers(0, len(y), len(y)) for _ in range(1000)]
boot = [b for b in boot if y[b].sum() > 0]

def ap(s, idx=None):
    return average_precision_score(y if idx is None else y[idx], s if idx is None else np.asarray(s)[idx])

scores = {f'단독: {k}': (rank01(v)) for k, v in vc.items()}
scores.update({k: blend(vc, w) for k, w in BLENDS.items()})
base = scores['B0_05방식(lr_base+rule)']
res = []
for name, s in scores.items():
    diff = np.array([ap(s, b) - ap(base, b) for b in boot])
    res.append({'방법': name, 'PR-AUC': round(ap(s), 4), 'ROC-AUC': round(roc_auc_score(y, s), 4),
                'B0 대비': round(ap(s) - ap(base), 4), 'P(개선)': round((diff > 0).mean(), 3)})
res = pd.DataFrame(res)
display(res)

plot = res.sort_values('PR-AUC')
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot['방법'], plot['PR-AUC'], color=['#4C72B0' if s.startswith('B') else '#A0A0A0' for s in plot['방법']])
ax.axvline(0.20, ls='--', c='gray')
for i, v in enumerate(plot['PR-AUC']):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center')
ax.set_xlim(0.15, max(0.55, plot['PR-AUC'].max() + 0.05)); ax.set_xlabel('08-25 홀드아웃 PR-AUC')
ax.set_title('검증 (학습: 자체 18일 라벨 07-26~08-07, 132h 마스킹)')
plt.tight_layout(); plt.show()

## 6. 최종 학습 · 09-01 예측 · 제출 파일 생성

- 모든 블렌드 후보의 제출 파일을 만든다. 최종 점수는 **제출한 것 중 최고점**이므로 여러 개를 내 보는 게 유리하다.
- 검증 1위를 `submit_06_best_*.csv`로 따로 저장한다.
- **동점 깨기**: `블렌드 순위 + 1e-6 × lr 확률 + 1e-9 × sim` 후 0~1로 정규화.

In [ ]:
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
test = sample[['row_id', 'channel_id']].merge(make_features(D, fut=FUT, past_days=PAST_DAYS), on='channel_id', how='left')
print('09-01 피처 결측 (log_past_med):', test.log_past_med.isna().sum(), '/', len(test))

tc = components(final_tr, test, D, FUT, ridge_train=own26)

def finalize(score):
    s = np.asarray(score) + 1e-6 * rank01(tc['lr']) + 1e-9 * rank01(tc['sim'])
    s = (s - s.min()) / (s.max() - s.min())
    sub = pd.DataFrame({'row_id': test.row_id, 'prediction': s})
    sub = sample[['row_id']].merge(sub, on='row_id', how='left')
    assert list(sub.columns) == ['row_id', 'prediction']
    assert len(sub) == len(sample)
    assert set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
    assert sub.prediction.notna().all() and np.isfinite(sub.prediction).all()
    assert sub.prediction.between(0, 1).all()
    return sub

ranking = res[res['방법'].str.startswith('B')].sort_values('PR-AUC', ascending=False).reset_index(drop=True)
saved = []
for i, r in ranking.iterrows():
    name = r['방법']
    sub = finalize(blend(tc, BLENDS[name]))
    tag = name.split('_')[0]
    path = f'{SUB_DIR}/submit_06_{tag}.csv'
    sub.to_csv(path, index=False, encoding='utf-8')
    if i == 0:
        sub.to_csv(f'{SUB_DIR}/submit_06_best_{tag}.csv', index=False, encoding='utf-8')
    saved.append({'제출 순서': i + 1, '파일': os.path.basename(path), '방법': name, '검증 PR-AUC': r['PR-AUC'],
                  '동점 행': int(sub.prediction.duplicated(keep=False).sum())})
display(pd.DataFrame(saved))

### 6.1. 후보끼리 얼마나 다른가

순위 상관이 매우 높은(>0.97) 후보들은 사실상 같은 제출이다. 제출 횟수가 남으면 **서로 덜 비슷한 후보**부터 내는 게 최고점을 뽑을 확률이 높다.

In [ ]:
from scipy.stats import spearmanr
names = list(BLENDS)
M = pd.DataFrame([[spearmanr(blend(tc, BLENDS[a]), blend(tc, BLENDS[b]))[0] for b in names] for a in names],
                 index=[n.split('_')[0] for n in names], columns=[n.split('_')[0] for n in names]).round(3)
M

## 7. 정리 (발표용)

| 단계 | 05번 | 06번 |
|---|---|---|
| view_5d 피처 | 기준일 직전 영상까지 사용 → **학습/예측 불일치** | 기준일 −132h 이전만 사용 (09-01 조건과 동일) |
| 라벨 재현 | 미검증 | 공식 08-11·08-25 행 집합·target과 대조 |
| 검증 | 26일 라벨 5일치로 학습 → 18일 라벨 평가 | 18일 라벨 13일치로 학습 → 18일 라벨 평가 |
| 신호 | 로지스틱 + 평균 회귀 규칙 | + 업로드율·무업로드 확률·상방 여지·출시 예정 게임, growth 회귀, 라벨 시뮬레이션 |
| 비교 | 단일 점수 | 부트스트랩으로 개선 확률 제시 |
| 제출 | 순위 동점 218행 | 동점 제거 |

**한계**: 홀드아웃이 한 시점(08-25, 666행)뿐이라 후보 간 0.01 수준 차이는 잡음일 수 있다. 26일 창을 깨끗하게 검증할 기준일이 데이터 범위상 없다.